# Vocal-aware chord analysis

BTC analyzes the original mix. HDemucs isolates vocals for CREPE, and brief BTC label flicker is smoothed before compatibility scoring.

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt

from reharmonize_models import (
    aggregate_melody_by_chord, detect_chords, melody_clash_score,
    melody_pitch_track, save_lab, separate_vocals, smooth_chord_intervals,
)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
audio_setting = os.environ.get('AUDIO_PATH')
AUDIO = Path(audio_setting) if audio_setting else ROOT / 'data/song.mp3'
if not AUDIO.is_absolute():
    AUDIO = ROOT / AUDIO
DURATION = float(os.environ.get('ANALYSIS_DURATION', '30'))
AUDIO, DURATION

In [ ]:
VOCALS = ROOT / 'artifacts' / f'{AUDIO.stem}-vocals.wav'
separate_vocals(AUDIO, VOCALS, duration=DURATION)
VOCALS

In [ ]:
raw_chords = detect_chords(AUDIO, btc_root=ROOT / 'vendor/BTC-ISMIR19', duration=DURATION)
chords = smooth_chord_intervals(raw_chords, minimum_duration=0.35)
save_lab(chords, ROOT / 'artifacts' / f'{AUDIO.stem}-smoothed.lab')
print({'raw_intervals': len(raw_chords), 'smoothed_intervals': len(chords)})
chords

In [ ]:
melody = melody_pitch_track(VOCALS, duration=DURATION, periodicity_threshold=0.5)
segments = aggregate_melody_by_chord(chords, melody)
segments['clash'] = segments.apply(
    lambda row: melody_clash_score(row.melody_profile, row.chord), axis=1
)
segments[['start', 'end', 'chord', 'voiced_frames', 'clash']]

In [ ]:
ax = melody.plot(x='time', y='midi', figsize=(14, 4), legend=False, title='Smoothed BTC chords over isolated vocal melody')
for row in chords.itertuples(index=False):
    ax.axvline(row.start, color='gray', alpha=0.2)
    ax.text((row.start + row.end) / 2, ax.get_ylim()[1], row.chord, ha='center', va='top', fontsize=8)
ax.set(xlabel='Time (seconds)', ylabel='MIDI pitch')
plt.show()